# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset title: {getattr(metadata, 'name', 'N/A')}")
print(f"Dataset description: {getattr(metadata, 'description', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We'll print the Record Sets, their `@id`, and for each Record Set, list the fields and their `@id`.

This helps to select the appropriate identifiers for subsequent extraction and analysis steps.

In [ ]:
# Fetch record sets and fields by @id
record_sets = dataset.record_sets
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet: {getattr(rs, '@id', 'N/A')}")
    print(f"  Name: {getattr(rs, 'name', getattr(rs, 'title', 'N/A'))}")
    fields = getattr(rs, 'fields', [])
    print(f"  Fields:")
    for field in fields:
        print(f"    - @id: {getattr(field, '@id', 'N/A')}, name: {getattr(field, 'name', 'N/A')}, data_type: {getattr(field, 'data_type', 'N/A')}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Let's select the main protein record set for further analysis. (The leading record set is typically the core protein information so we'll extract all and filter down.)

In [ ]:
# Extract data from each record set identified above
# (Update these IDs as needed based on the printed output)

# Gather the record set @id values
record_set_ids = [rs['@id'] if isinstance(rs, dict) else getattr(rs, '@id', None) for rs in dataset.record_sets]

# Filter out None values
record_set_ids = [rsid for rsid in record_set_ids if rsid is not None]

dataframes = dict()

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for RecordSet @id: {record_set_id}")
    except Exception as e:
        print(f"Failed loading RecordSet {record_set_id}: {e}")

# Show the field (column) list for the first non-empty record set
for rsid, df in dataframes.items():
    if not df.empty:
        print(f"\nSample columns for RecordSet @id {rsid}:")
        print(df.columns.tolist())
        display(df.head())
        # Store for next steps
        main_record_set_id = rsid
        break

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.
This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Automatically select a numeric field for demonstration
import numpy as np

df = dataframes[main_record_set_id]
# Prefer columns with molecular weight, abundance, coverage, count, etc.
preferred_numeric_fields = [col for col in df.columns if any(key in col.lower() for key in ['coverage', 'mw', 'abundance', 'count', 'peptide', 'score', 'number', 'molecular', 'normalized'])]
if preferred_numeric_fields:
    numeric_field = preferred_numeric_fields[0]
else:
    # Fallback: Pick the first column with numeric data
    possible_numeric = df.select_dtypes(include=np.number).columns.tolist()
    numeric_field = possible_numeric[0] if possible_numeric else df.columns[0]  # at least something

print(f"Analyzing numeric field: {numeric_field}")

# Try to convert if needed
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

# Remove outliers (values < 1st or > 99th percentile)
q_low = df[numeric_field].quantile(0.01)
q_high = df[numeric_field].quantile(0.99)
filtered_df = df[(df[numeric_field] > q_low) & (df[numeric_field] < q_high)]
print(f"Filtered outliers, kept {len(filtered_df)} of {len(df)} rows.")

# Normalize field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a categorical field
possible_cats = [col for col in df.columns if col != numeric_field and df[col].dtype == object and df[col].nunique() < 20]
group_field = possible_cats[0] if possible_cats else None
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index().sort_values(numeric_field, ascending=False)
    print(f"\nMean {numeric_field} grouped by '{group_field}':")
    print(grouped_df)

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below, we plot the distribution for the selected numeric field, grouped by the chosen categorical variable if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 4))
sns.histplot(filtered_df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

if group_field:
    plt.figure(figsize=(10, 6))
    sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we have loaded and explored the FAIR² mass spectrometry dataset using the `mlcroissant` library. We listed its record sets, extracted data using entity `@id` references, filtered and normalized a numeric field, and visualized distributions and groupings. This workflow helps ensure reproducible and standard data handling for Croissant-based datasets.